In [ ]:
# ---- Paths ----
BASE_DIR = os.environ.get("BASE_DIR", "/kaggle/working")
REPORTS_DIR = os.environ.get("REPORTS_DIR", "/kaggle/input/datasets/zz4825/v5-newlabel")
GT_DIR = os.environ.get("GT_DIR", "/kaggle/input/datasets/zz4825/v5-newlabel")
SCHEMA_39_PATH = os.environ.get("SCHEMA_39_PATH", "/kaggle/input/datasets/zz4825/schema/extraction_schema_2025_sc_39new.json")
PRED_DIR      = os.environ.get("PRED_DIR",      os.path.join(BASE_DIR, "predictions"))
EVAL_FIG_DIR  = os.environ.get("EVAL_FIG_DIR",  os.path.join(BASE_DIR, "eval_figs"))
EVAL_CONF_DIR = os.environ.get("EVAL_CONF_DIR", os.path.join(BASE_DIR, "eval_confusions"))
  

AUTO_MKDIRS = os.environ.get("AUTO_MKDIRS", "True").lower() in {"1","true","t","yes","y","on"}

# ---- Model download/cache control ----
ALLOW_HF_DOWNLOAD = os.environ.get("ALLOW_HF_DOWNLOAD", "True").lower() in {"1","true","t","yes","y","on"}

HF_CACHE_DIR = os.environ.get("HF_CACHE_DIR", os.path.join(BASE_DIR, "hf_cache"))
os.makedirs(HF_CACHE_DIR, exist_ok=True)

os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", os.environ.get("HF_HUB_ENABLE_HF_TRANSFER", "1"))

MODEL_CANDIDATES = json.loads(os.environ.get(
    "MODEL_CANDIDATES_JSON",
    '{"qwen1.5b":"Qwen/Qwen2.5-1.5B-Instruct"}'
))

MODELS_TO_RUN = os.environ.get("MODELS_TO_RUN", "qwen1.5b").split(",")

# ---- Experiment switches ----
EXPERIMENT_MODE = os.environ.get("EXPERIMENT_MODE", "both")  # zero_shot | fine_tune | both

# ---- Repro / splitting ----
SEED = int(os.environ.get("SEED", "42"))
TRAIN_FRAC = float(os.environ.get("TRAIN_FRAC", "0.80"))
VAL_FRAC   = float(os.environ.get("VAL_FRAC",   "0.10"))
TEST_FRAC  = float(os.environ.get("TEST_FRAC",  "0.10"))

MAX_SAMPLES_RAW = os.environ.get("MAX_SAMPLES", "")
MAX_SAMPLES = int(MAX_SAMPLES_RAW) if MAX_SAMPLES_RAW else None

# ---- Lengths ----
MAX_LEN_INPUT  = int(os.environ.get("MAX_LEN_INPUT", "4096"))
MAX_LEN_OUTPUT = int(os.environ.get("MAX_LEN_OUTPUT", "512"))

# ---- Training HP ----
BATCH_SIZE_TRAIN  = int(os.environ.get("BATCH_SIZE_TRAIN", "1"))
GRAD_ACCUM_STEPS  = int(os.environ.get("GRAD_ACCUM_STEPS", "16"))
LR                = float(os.environ.get("LR", "2e-4"))
NUM_EPOCHS        = int(os.environ.get("NUM_EPOCHS", "15"))
WARMUP_RATIO      = float(os.environ.get("WARMUP_RATIO", "0.03"))
WEIGHT_DECAY      = float(os.environ.get("WEIGHT_DECAY", "0.0"))
LOG_STEPS         = int(os.environ.get("LOG_STEPS", "5"))
SAVE_STEPS        = int(os.environ.get("SAVE_STEPS", "200"))
EVAL_STEPS        = int(os.environ.get("EVAL_STEPS", "200"))

# ---- LoRA ----
LORA_R       = int(os.environ.get("LORA_R", "8"))
LORA_ALPHA   = int(os.environ.get("LORA_ALPHA", "16"))
LORA_DROPOUT = float(os.environ.get("LORA_DROPOUT", "0.05"))

TARGET_MODULES = os.environ.get(
    "TARGET_MODULES",
    "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj"
).split(",")

# ---- Multi-run control ----
N_RUNS = int(os.environ.get("N_RUNS", "3"))
_RUN_SEEDS_JSON = os.environ.get("RUN_SEEDS_JSON")
RUN_SEEDS = json.loads(_RUN_SEEDS_JSON) if _RUN_SEEDS_JSON is not None else [SEED + i for i in range(N_RUNS)]

# ---- Jsonformer strictness ----
ENFORCE_JSONFORMER = os.environ.get("ENFORCE_JSONFORMER", "True").lower() in {"1","true","t","yes","y","on"}
JSONFORMER_MAX_STRING_TOKEN_LENGTH = int(os.environ.get("JSONFORMER_MAX_STRING_TOKEN_LENGTH", "128"))

# ---- Run tag / naming ----
RUN_TAG_FMT = os.environ.get("RUN_TAG_FMT", "sm_ft_%Y%m%d_%H%M%S")
RUN_TAG = os.environ.get("RUN_TAG")
if RUN_TAG is None:
    RUN_TAG = datetime.datetime.now().strftime(RUN_TAG_FMT)

# ---- Create output dirs (optional) ----
if AUTO_MKDIRS:
    for d in (PRED_DIR, EVAL_FIG_DIR, EVAL_CONF_DIR):
        os.makedirs(d, exist_ok=True)


In [ ]:
!pip install numpy==1.24.3
!pip install pandas scipy==1.10.1 scikit-learn matplotlib
!pip install transformers==4.44.2 accelerate peft datasets huggingface_hub
!pip install jsonformer jsonschema pdfminer.six python-docx
!pip install bitsandbytes triton

from jsonformer.main import Jsonformer
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import sys, re, json, pathlib, gc, random, hashlib, datetime, time, importlib.metadata as _md
import math as _math
from dataclasses import dataclass
from inspect import signature
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-muted')

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
from huggingface_hub import login, snapshot_download
from jsonschema import Draft7Validator, validators as _js_validators

from jsonformer.format import highlight_values
from jsonformer.main import Jsonformer

from transformers import EarlyStoppingCallback

import bitsandbytes as bnb
print(f"bitsandbytes version: {bnb.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'None'}")

#!pip install pdfminer.six python-docx
from pdfminer.high_level import extract_text
import docx
from IPython.display import display

In [ ]:
# ============================ STRICT schema: load from JSON file ==========================

def load_json_schema(path: str):
    """Load JSON Schema from disk + build the correct validator for its declared draft."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"[ERROR] Schema file not found: {path}")

    with open(path, "r", encoding="utf-8") as f:
        schema = json.load(f)

    ValidatorCls = _js_validators.validator_for(schema)
    ValidatorCls.check_schema(schema)
    validator = ValidatorCls(schema)
    return schema, validator

# Load the 39-key inner schema
ENDOSCOPY_SCHEMA, ENDO_VALIDATOR = load_json_schema(SCHEMA_39_PATH)

# ========== 2. 新增：加载别名映射文件 ==========
ALIAS_MAP_PATH = os.environ.get("ALIAS_MAP_PATH", "/kaggle/input/datasets/zz4825/qwen-alias/qwen_output_alias_map4_1.json")

def load_alias_map(path: str) -> dict:
    if not os.path.exists(path):
        print(f"[WARN] Alias map not found: {path}, using empty mapping.")
        return {"global_value_aliases": {}, "field_name_aliases": {}, "field_value_aliases": {}}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

ALIAS_MAP = load_alias_map(ALIAS_MAP_PATH)
# 这些字段不应用值别名映射（保留原始值，由后续逻辑专门处理）
NUMERIC_LIKE_FIELDS = {"neoplastic_lesion_size", "pepsinogen_i_level", "pepsinogen_ii_level", "patient_age"}

def apply_alias_map(value: Any, key: str) -> str:
    if not isinstance(value, str):
        value = str(value)
    original = value.strip()
    if original == "":
        return "not_mentioned"
    
    # 数值/单位字段不应用别名映射，避免破坏格式
    if key in NUMERIC_LIKE_FIELDS:
        return original
    
    # 1. 字段特定值别名
    field_aliases = ALIAS_MAP.get("field_value_aliases", {}).get(key, {})
    if original in field_aliases:
        mapped = field_aliases[original]
        return mapped if mapped is not None else "not_mentioned"
    
    # 2. 全局值别名
    global_aliases = ALIAS_MAP.get("global_value_aliases", {})
    if original in global_aliases:
        mapped = global_aliases[original]
        return mapped if mapped is not None else "not_mentioned"
    
    return original



# Derive REQUIRED_KEYS / enums / patterns from the schema file
REQUIRED_KEYS = list(ENDOSCOPY_SCHEMA.get("required", []))

_props = ENDOSCOPY_SCHEMA.get("properties", {})
ALLOWED_ENUMS = {k: v["enum"] for k, v in _props.items() if isinstance(v, dict) and "enum" in v}
PATTERNS      = {k: v["pattern"] for k, v in _props.items() if isinstance(v, dict) and "pattern" in v}

# "Sanity" checks without printing keys/values
if not REQUIRED_KEYS:
    raise ValueError("[ERROR] Schema has no 'required' keys.")
if len(REQUIRED_KEYS) != len(set(REQUIRED_KEYS)):
    raise ValueError("[ERROR] Schema 'required' contains duplicates.")
if ENDOSCOPY_SCHEMA.get("additionalProperties", None) is not False:
    print("[WARN] Schema additionalProperties is not false. Your file says false; double-check schema.")

print(f"[SCHEMA] Loaded: {SCHEMA_39_PATH} | n_required={len(REQUIRED_KEYS)}")

# ---------- Combined (single-set) schema ----------
COMBINED_SINGLE_SCHEMA = {
    "type": "object",
    "properties": {
        "patient_id": {"type": "string"},
        "variant": {"type": "string", "enum": ["correct", "wrong"]},
        "endoscopy and pathology": ENDOSCOPY_SCHEMA,  # <-- loaded from file
    },
    "required": ["patient_id", "variant", "endoscopy and pathology"],
    "additionalProperties": False,
}

# ============================ Prompts =======================================
def _schema_to_guide(schema: dict) -> dict:
    props = schema.get("properties", {})
    req   = schema.get("required", list(props.keys()))
    guide = {}
    for k in req:
        spec = props.get(k, {})
        if isinstance(spec, dict) and "enum" in spec:
            guide[k] = spec["enum"]
        elif isinstance(spec, dict) and "pattern" in spec:
            guide[k] = "pattern:" + spec["pattern"]
        else:
            guide[k] = "short text / not_mentioned"
    return guide

SCHEMA_GUIDE = _schema_to_guide(ENDOSCOPY_SCHEMA)

def make_joint_prompt_single(pid: str, variant: str, endo_text: str, path_text: str) -> str:
    return f"""
You are given two reports for the SAME patient: an Endoscopy report and a Pathology report.
Extract ONE unified set of structured findings using BOTH reports as evidence.

Return ONLY a single JSON object with this exact structure:
{{
  "patient_id": "{pid}",
  "variant": "{variant}",
  "endoscopy and pathology": {{ ... EXACT keys below ... }}
}}

Rules:
- Use EXACT spellings for keys and allowed values.
- If a finding is not reported in either report, set "not_mentioned.
- No explanations. No extra keys.

Keys and allowed values (loaded from JSON schema file):
{json.dumps(SCHEMA_GUIDE, indent=2)}

--- ENDOSCOPY REPORT ---
{endo_text}

--- PATHOLOGY REPORT ---
{path_text}
""".strip()

print(f"[SCHEMA] file={SCHEMA_39_PATH}")
print(f"[SCHEMA] required_count={len(REQUIRED_KEYS)}")
print(f"[SCHEMA] has_enums={sum(1 for k in REQUIRED_KEYS if k in ALLOWED_ENUMS)}")
print(f"[SCHEMA] has_patterns={sum(1 for k in REQUIRED_KEYS if k in PATTERNS)}")

# ============================ Validation & sanitization ======================

def _norm_enum_text(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", s.strip().lower())

_NOT_MENTIONED_NORM = {_norm_enum_text(x) for x in [
    "not_mentioned", "not mentioned", "notmentioned",
    "n/a", "na", "none", "unknown", "-", "null", ""
]}

# enum normalization map: per key, map normalized text -> canonical allowed enum value
_ENUM_NORM_MAP: Dict[str, Dict[str, str]] = {
    k: {_norm_enum_text(v): v for v in vals} for k, vals in ALLOWED_ENUMS.items()
}

_NUM_RE = re.compile(r"(\d+(?:\.\d+)?)")

def _coerce_to_allowed(key: str, value: Any) -> str:
    import re
    if value is None:
        return "not_mentioned"

    s = str(value).strip()
    if not s or _norm_enum_text(s) in _NOT_MENTIONED_NORM:
        return "not_mentioned"

    # --- 特殊数值字段处理（保持不变） ---
    if key == "neoplastic_lesion_size":
        m = re.search(r"(\d+(?:\.\d+)?)\s*(mm|cm)?", s, flags=re.I)
        if not m:
            return "not_mentioned"
        val = float(m.group(1))
        unit = (m.group(2) or "mm").lower()
        if unit == "cm":
            val *= 10.0
        val_str = f"{val:.3f}".rstrip("0").rstrip(".")
        return f"{val_str} mm"

    if key == "pepsinogen_i_level":
        m = _NUM_RE.search(s)
        return f"{m.group(1)} ng/ml" if m else "not_mentioned"

    if key == "pepsinogen_ii_level":
        m = _NUM_RE.search(s)
        return f"{m.group(1)} ng/ml" if m else "not_mentioned"

    # ================= 新增：OLGA / OLGIM 特殊处理 =================
    if key in ["olga_stage", "olgim_stage"]:
        s_lower = s.lower()
        # 1. 直接数字 "0","1","2","3","4"
        digit_map = {"0": "0", "1": "i", "2": "ii", "3": "iii", "4": "iv"}
        if s_lower in digit_map:
            return digit_map[s_lower]
        # 2. 带 "stage" 前缀，如 "stage 2", "stage ii"
        import re
        m = re.search(r'stage\s*([0-4]|i{1,3}|iv)', s_lower)
        if m:
            sub = m.group(1)
            if sub in digit_map:
                return digit_map[sub]
            if sub in ["i", "ii", "iii", "iv"]:
                return sub
        # 3. 纯罗马数字 "ii"、"III" 等（已小写）
        m = re.search(r'\b(i{1,3}|iv)\b', s_lower)
        if m:
            return m.group(1)
        # 4. 兜底：如果以上都不匹配，返回 not_mentioned
        return "not_mentioned"


    # --- enums: accept formatting differences ---
    if key in ALLOWED_ENUMS:
        if s in ALLOWED_ENUMS[key]:
            return s
        norm = _norm_enum_text(s)
        mapped = _ENUM_NORM_MAP[key].get(norm)
        return mapped if mapped is not None else "not_mentioned"

    # --- patterns / free text fallback ---
    if key in PATTERNS:
        return s if re.match(PATTERNS[key], s) else "not_mentioned"

    return s[:128] if len(s) > 128 else s


def validate_and_sanitize(obj: Dict[str, Any], validator: Draft7Validator) -> Dict[str, str]:
    # 字段名别名重命名
    if isinstance(obj, dict):
        field_aliases = ALIAS_MAP.get("field_name_aliases", {})
        renamed = {}
        for k, v in obj.items():
            target_key = field_aliases.get(k, k)
            renamed[target_key] = v
        obj = renamed
    
    cleaned = {k: "not_mentioned" for k in REQUIRED_KEYS}
    if isinstance(obj, dict):
        for k in REQUIRED_KEYS:
            if k in obj:
                raw_val = obj[k]
                aliased_val = apply_alias_map(raw_val, k)
                cleaned[k] = _coerce_to_allowed(k, aliased_val)
    
    # 最终验证兜底
    for error in validator.iter_errors(cleaned):
        path_key = list(error.path)[0] if error.path else None
        if path_key in cleaned:
            cleaned[path_key] = "not_mentioned"
    return cleaned


# ============================ File readers ==================================
def _read_txt_like(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def _read_pdf(path: str) -> str:
    try:
        return extract_text(path) or ""
    except Exception as e:
        print(f"[WARN] PDF extract failed for {path}: {e}")
        return ""

def _read_docx(path: str) -> str:
    try:
        doc = docx.Document(path)
        return "\n".join(p.text for p in doc.paragraphs)
    except Exception as e:
        print(f"[WARN] DOCX read failed for {path}: {e}")
        return ""

def read_report_file(path: Optional[str]) -> str:
    if not path:
        return ""
    ext = pathlib.Path(path).suffix.lower()
    if ext in {".txt", ".md"}:
        content = _read_txt_like(path)
    elif ext == ".pdf":
        content = _read_pdf(path)
    elif ext == ".docx":
        content = _read_docx(path)
    else:
        try:
            content = _read_txt_like(path)
        except Exception:
            return ""
    
    # ---- 剥离标题行 ----
    lines = content.splitlines()
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        # 跳过以 "===" 开头或包含 "(Correct)" 或 "(Wrong)" 的行
        if stripped.startswith("==="):
            continue
        if "(Correct)" in stripped or "(Wrong)" in stripped:
            continue
        cleaned_lines.append(line)
    return "\n".join(cleaned_lines)


# ============================ Discovery helpers =============================
MOD_RE = re.compile(r"_(Endoscopy|Pathology)(?=[_\.\-\s]|$)", re.I)
VARIANT_RE = re.compile(r"_(correct|wrong)(?=[_\.\-\s]|$)", re.I)
SUPPORTED_EXT = {".txt", ".md", ".pdf", ".docx"}

import re
from pathlib import Path
from typing import Optional, Tuple

# 黑名单：仅排除明确不需要的关键词
EXCLUDE_KEYWORDS = {"explanation", "prompt", "master"}

def parse_report_filename(fname: str) -> Optional[Tuple[str, str, str]]:
    fname_lower = fname.lower()
    
    # 1. 黑名单过滤
    if any(kw in fname_lower for kw in EXCLUDE_KEYWORDS):
        return None

    ext = Path(fname).suffix.lower()
    if ext not in SUPPORTED_EXT:   # SUPPORTED_EXT 在前文定义
        return None

    # 2. 模态匹配：寻找第一个 _Endoscopy 或 _Pathology
    mod_match = re.search(r"_(Endoscopy|Pathology)(?=[_.]|$)", fname, re.I)
    if not mod_match:
        return None

    # 3. 变体匹配：收集所有 _correct 或 _wrong，取最后一个
    var_matches = list(re.finditer(r"_(correct|wrong)(?=[_.]|$)", fname, re.I))
    if not var_matches:                # ← 这里增加了条件判断
        return None                    # ← 缩进已修正，与上一行 if 对齐
    var_match = var_matches[-1]        # 取最后一个

    # 4. 确保模态在变体之前（防止如 _wrong_Endoscopy 这种错误）
    if mod_match.end() > var_match.start():
        return None

    pid = fname[:mod_match.start()]
    if not pid:
        return None

    variant = var_match.group(1).lower()
    modality = "endoscopy" if mod_match.group(1).lower() == "endoscopy" else "pathology"
    return pid, variant, modality


# ============================ Generation helper =============================
JSONFORMER_CALLS = 0

def run_jsonformer(tokenizer, model, schema: dict, prompt: str, max_string_token_length: int = 128):
    global JSONFORMER_CALLS
    JSONFORMER_CALLS += 1
    try:
        builder = Jsonformer(
            model=model,
            tokenizer=tokenizer,
            json_schema=schema,
            prompt=prompt,
            max_string_token_length=max_string_token_length,
            temperature=1e-6
        )
        out = builder()
        try:
            highlight_values(out)
        except Exception:
            pass
        return out
    except Exception as e:
        print(f"[WARN] Jsonformer failed: {e}")
        default_inner = {k: "not_mentioned" for k in REQUIRED_KEYS}
        return {
            "patient_id": "unknown",
            "variant": "unknown",
            "endoscopy and pathology": default_inner
        }

_CANON_MAP = {
    "non-visible": "nonvisible",
    "non visible": "nonvisible",
    "nonsmoker": "non_smoker",
    "yes": "yes",
    "no": "no",
    "not mentioned": "not_mentioned",
    "not_mentioned": "not_mentioned",
    "na": "not_mentioned",
    "none": "not_mentioned",
    "unknown": "not_mentioned",
}

def _canon_str(x: str) -> str:
    if not isinstance(x, str):
        return str(x)
    s = x.strip()
    return _CANON_MAP.get(s.lower(), s)


def _norm_lesion_size_mm(s):
    if not isinstance(s, str):
        return None
    t = s.strip()
    if t.lower() in ("not_mentioned", "not mentioned"):
        return None
    m = re.search(r"(\d+(?:\.\d+)?)\s*(mm|cm)?", t, re.I)
    if not m:
        return None
    val = float(m.group(1))
    unit = (m.group(2) or "mm").lower()
    if unit == "cm":
        val *= 10.0
    return val


def _extract_number(s):
    if not isinstance(s, str):
        return None
    t = s.strip()
    if t.lower() in ("not_mentioned", "not mentioned"):
        return None
    m = re.search(r"(\d+(?:\.\d+)?)", t)
    return float(m.group(1)) if m else None


def values_equal(k, y_true, y_pred, tol=1e-3):
    if k == "neoplastic_lesion_size":
        t = _norm_lesion_size_mm(y_true)
        p = _norm_lesion_size_mm(y_pred)
        if t is None and p is None:
            return True
        if (t is None) != (p is None):
            return False
        return abs(t - p) <= tol

    if k in ["pepsinogen_i_level", "pepsinogen_ii_level"]:
        t = _extract_number(y_true)
        p = _extract_number(y_pred)
        if t is None and p is None:
            return True
        if (t is None) != (p is None):
            return False
        return abs(t - p) <= tol

    yt = _canon_str(str(y_true) if y_true is not None else "not_mentioned") or "not_mentioned"
    yp = _canon_str(str(y_pred) if y_pred is not None else "not_mentioned") or "not_mentioned"
    return yt == yp

from pathlib import Path  
def evaluate_and_plot(gt_dir: str = GT_DIR, pred_dir: str = PRED_DIR,
                      fig_dir: str = EVAL_FIG_DIR, conf_dir: str = EVAL_CONF_DIR):
    if not os.path.isdir(gt_dir):
        raise SystemExit(f"[ERROR] Ground-truth directory not found: {gt_dir}")

    # ---------- 递归查找所有 GT JSON ----------
    gt_path = Path(gt_dir)
    gt_files = [str(f) for f in gt_path.glob("**/*_extracted_*.json")]
    if not gt_files:
        raise SystemExit(f"[ERROR] No ground-truth JSONs found in {gt_dir}. "
                         "Expected '*_extracted_correct.json' / '*_extracted_wrong.json' "
                         "in any subfolder.")

    # ---------- 辅助函数 ----------
    def infer_pid_variant_from_gt(full_path: str):
        basename = os.path.basename(full_path)
        m = re.match(r"^(?P<pid>.*)_extracted_(?P<variant>correct|wrong)\.json$", basename, flags=re.I)
        return (m.group("pid"), m.group("variant").lower()) if m else (None, None)

    def _normalize_to_single(obj: dict) -> Dict[str, str]:
        """
        Normalize GT or prediction to a single dict of REQUIRED_KEYS.
        Prefer formats:
          1) {"endoscopy and pathology": {...}}
          2) flat at root (all REQUIRED_KEYS present)
          3) legacy {"endoscopy": {...}, "pathology": {...}} -> pathology > endoscopy fallback
        """
        if isinstance(obj, dict):
            if "endoscopy and pathology" in obj and isinstance(obj["endoscopy and pathology"], dict):
                return ensure_all_keys(obj["endoscopy and pathology"])
            if all(k in obj for k in REQUIRED_KEYS):
                return ensure_all_keys(obj)
            if "endoscopy" in obj or "pathology" in obj:
                endo = ensure_all_keys(obj.get("endoscopy", {}))
                path = ensure_all_keys(obj.get("pathology", {}))
                return {k: (path[k] if path[k] != "not_mentioned" else endo[k]) for k in REQUIRED_KEYS}
        return {k: "not_mentioned" for k in REQUIRED_KEYS}

    def read_flat_gt(gt_full_path: str) -> Dict[str, str]:
        return _normalize_to_single(_read_json(gt_full_path) or {})

    def read_flat_pred(pid: str, variant: str) -> Dict[str, str] | None:
        pp = os.path.join(pred_dir, f"{pid}_{variant}_extracted_pred.json")
        if not os.path.exists(pp):
            return None
        return _normalize_to_single(_read_json(pp) or {})

    rows, missing_pred = [], []
    # ----------循环使用完整路径 ----------
    for gt_full_path in sorted(gt_files):
        pid, variant = infer_pid_variant_from_gt(gt_full_path)
        if not pid:
            continue
        gt_flat = read_flat_gt(gt_full_path)
        pred_flat = read_flat_pred(pid, variant)
        if pred_flat is None:
            missing_pred.append(f"{pid}:{variant}")
            continue
        for feat in REQUIRED_KEYS:
            y_true = gt_flat.get(feat, "not_mentioned")
            y_pred = pred_flat.get(feat, "not_mentioned")
            rows.append({
                "patient_id": pid,
                "variant": variant,
                "feature": feat,
                "y_true": str(y_true),
                "y_pred": str(y_pred),
                "correct": int(values_equal(feat, y_true, y_pred)),
            })

    df = pd.DataFrame(rows)
    if df.empty:
        raise SystemExit(f"[ERROR] No comparable GT/Pred pairs. Missing preds for: {missing_pred}")

    overall_acc = df["correct"].mean()
    feat_acc = (df.groupby("feature")["correct"].mean()
                  .sort_values(ascending=False).rename("accuracy").reset_index())
    pat_acc = (df.groupby(["patient_id","variant"])["correct"].mean()
                 .rename("accuracy").reset_index()
                 .sort_values(["patient_id","variant"], ascending=True))

    # Macro-F1 per feature
    def macro_f1(sub):
        labels = sorted(set(sub["y_true"]) | set(sub["y_pred"]))
        cm = {(t,p):0 for t in labels for p in labels}
        for _,r in sub.iterrows():
            cm[(r["y_true"], r["y_pred"])] += 1
        f1s=[]
        for c in labels:
            tp = cm[(c,c)]
            fp = sum(cm[(t,c)] for t in labels if t!=c)
            fn = sum(cm[(c,p)] for p in labels if p!=c)
            prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
            rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
            f1s.append((2*prec*rec)/(prec+rec) if (prec+rec)>0 else 0.0)
        return float(np.mean(f1s)) if f1s else 0.0

    macro = (df.groupby("feature").apply(macro_f1).rename("macro_f1").reset_index())
    feat_scores = feat_acc.merge(macro, on="feature", how="left")

    # 非空准确率
    df_non_null = df[df["y_true"] != "not_mentioned"]
    non_null_overall = df_non_null["correct"].mean() if len(df_non_null) > 0 else float('nan')
    non_null_feat = (df_non_null.groupby("feature")["correct"]
                     .mean()
                     .rename("non_null_accuracy")
                     .reset_index())
    feat_scores = feat_scores.merge(non_null_feat, on="feature", how="left")
   # ---- 计算空值准确率 ----
    df_null = df[df["y_true"] == "not_mentioned"]
    null_accuracy = df_null["correct"].mean() if len(df_null) > 0 else float('nan')

    # ---- 计算平衡准确率（字段级宏平均） ----
    from sklearn.metrics import balanced_accuracy_score
    field_balanced_list = []
    for field, group in df.groupby('feature'):
        # 如果该字段只有一个类别，无法计算 balanced accuracy，跳过
        if len(group['y_true'].unique()) >= 2:
            bal = balanced_accuracy_score(group['y_true'], group['y_pred'])
            field_balanced_list.append(bal)
    balanced_accuracy = np.nanmean(field_balanced_list) if field_balanced_list else np.nan


    # Confusion matrices
    os.makedirs(conf_dir, exist_ok=True)
    for feat, sub in df.groupby("feature"):
        labels = sorted(set(sub["y_true"]) | set(sub["y_pred"]))
        mat = pd.DataFrame(0, index=labels, columns=labels, dtype=int)
        for _,r in sub.iterrows():
            mat.loc[r["y_true"], r["y_pred"]] += 1
        safe = re.sub(r'[^A-Za-z0-9]+','_',feat)
        mat.to_csv(os.path.join(conf_dir, f"confusion_{safe}.csv"))

    # Plots
    os.makedirs(fig_dir, exist_ok=True)

    # 1) Per-feature accuracy
    plt.figure(figsize=(10, max(4, 0.28*len(feat_acc))))
    plt.barh(feat_acc["feature"], feat_acc["accuracy"])
    plt.xlabel("Accuracy")
    plt.title("Per-feature accuracy (single-set from both reports)")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "per_feature_accuracy.png"), dpi=180)
    plt.close()

    # 2) Per-patient accuracy by variant
    wide = pat_acc.pivot(index="patient_id", columns="variant", values="accuracy").sort_index()
    for v in ("correct","wrong"):
        if v not in wide.columns:
            wide[v] = np.nan
    idx = np.arange(len(wide))
    w = 0.4
    plt.figure(figsize=(10, max(3, 0.4*len(wide))))
    plt.barh(idx - w/2, wide["correct"], height=w, label="correct")
    plt.barh(idx + w/2, wide["wrong"],   height=w, label="wrong")
    plt.yticks(idx, wide.index)
    plt.xlabel("Accuracy")
    plt.title("Per-patient accuracy by variant (single-set)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "per_patient_accuracy_by_variant.png"), dpi=180)
    plt.close()

    # 3) Non-null accuracy
    plt.figure(figsize=(10, max(4, 0.28*len(feat_scores))))
    plt.barh(feat_scores["feature"], feat_scores["non_null_accuracy"].fillna(0))
    plt.xlabel("Non-null Accuracy")
    plt.title("Per-feature non-null accuracy (only fields mentioned in GT)")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "per_feature_non_null_accuracy.png"), dpi=180)
    plt.close()

    print(f"\nOverall accuracy (pooled): {overall_acc:.3f}")
    print(f"Non-null overall accuracy: {non_null_overall:.3f}")
    print(f"GT dir : {gt_dir}")
    print(f"Pred dir: {pred_dir}")
    print(f"Confusions: {conf_dir}")
    print(f"Figures  : {fig_dir}")

    try:
        display(feat_scores.sort_values("accuracy", ascending=False).reset_index(drop=True))
        display(pat_acc)
    except Exception:
        pass

    return {
        "overall_accuracy": overall_acc,
        "non_null_overall_accuracy": non_null_overall,
        "null_accuracy": null_accuracy,            
        "balanced_accuracy": balanced_accuracy,    
        "feature_scores": feat_scores,
        "patient_scores": pat_acc,
        "raw": df
    }

In [ ]:
_mode = EXPERIMENT_MODE.strip().lower().replace("-", "_").replace(" ", "_")
if _mode in {"zs", "zero", "zero_shot", "zeroshot"}:
    _mode = "zero_shot"
elif _mode in {"ft", "finetune", "fine_tune", "fine_tuning"}:
    _mode = "fine_tune"
elif _mode in {"both", "all"}:
    _mode = "both"
else:
    raise ValueError(...)
DO_ZERO_SHOT = _mode in {"zero_shot", "both"}
DO_FINE_TUNE = _mode in {"fine_tune", "both"}

print(f"[RUN-MODE] EXPERIMENT_MODE='{EXPERIMENT_MODE}' → normalized='{_mode}' | "
      f"DO_ZERO_SHOT={DO_ZERO_SHOT} | DO_FINE_TUNE={DO_FINE_TUNE}")

if not os.path.isdir(REPORTS_DIR):
    raise SystemExit(f"[ERROR] Reports directory not found: {REPORTS_DIR}")

# -------------------------------- Utilities --------------------------------

def _feature_col(df: pd.DataFrame) -> str:
    for c in ["feature", "field", "key", "name", "Feature", "Field"]:
        if c in df.columns: return c
    return df.columns[0]

def _read_json(path):

  with open(path, "r", encoding="utf-8") as f:

    return json.load(f)

def ensure_all_keys(d: Any) -> Dict[str, str]:
    out = {k: "not_mentioned" for k in REQUIRED_KEYS}
    if isinstance(d, dict):
        out.update({k: d[k] for k in d.keys() & out.keys()})
    return out

_ensure_all_keys = ensure_all_keys

def normalize_to_single(obj: Any) -> Dict[str, str]:
    """
    Normalize GT or prediction to a single dict of REQUIRED_KEYS.
    Prefer formats:
      1) {"endoscopy and pathology": {...}}
      2) flat at root (all REQUIRED_KEYS present)
      3) legacy {"endoscopy": {...}, "pathology": {...}} -> pathology > endoscopy fallback
    """
    if isinstance(obj, dict):
        if "endoscopy and pathology" in obj and isinstance(obj["endoscopy and pathology"], dict):
            return ensure_all_keys(obj["endoscopy and pathology"])

        if all(k in obj for k in REQUIRED_KEYS):
            return ensure_all_keys(obj)

        if "endoscopy" in obj or "pathology" in obj:
            endo = ensure_all_keys(obj.get("endoscopy", {}))
            path = ensure_all_keys(obj.get("pathology", {}))
            return {k: (path[k] if path[k] != "not_mentioned" else endo[k]) for k in REQUIRED_KEYS}

    return {k: "not_mentioned" for k in REQUIRED_KEYS}

def collect_patient_variant_files(reports_dir: str) -> Dict[Tuple[str, str], Dict[str, str]]:
    pfiles: Dict[Tuple[str, str], Dict[str, str]] = {}
    for root, _, files in os.walk(reports_dir):
        for fn in files:
            parsed = parse_report_filename(fn)
            if not parsed:
                continue
            pid, variant, modality = parsed
            full = os.path.join(root, fn)
            key = (pid, variant)
            pfiles.setdefault(key, {})
            prev = pfiles[key].get(modality)
            if prev is None or os.path.getmtime(full) > os.path.getmtime(prev):
                pfiles[key][modality] = full
    return pfiles

def set_seed_all(seed: int = SEED):
    random.seed(seed); np.random.seed(seed % (2**32))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def _schema_sig(schema: dict) -> str:
    return hashlib.sha1(json.dumps(schema, sort_keys=True).encode()).hexdigest()[:10]

def format_target_json(pid: str, variant: str, flat_dict: Dict[str,str]) -> str:
    obj = {"patient_id": pid, "variant": variant,
           "endoscopy and pathology": {k: str(flat_dict.get(k, "Not Mentioned")) for k in REQUIRED_KEYS}}
    return json.dumps(obj, ensure_ascii=False, indent=2)

def apply_chat(tok, prompt_text: str) -> str:
    try:
        return tok.apply_chat_template(
            [{"role":"user","content":prompt_text}],
            tokenize=False, add_generation_prompt=True
        )
    except Exception:
        return prompt_text


def build_io_pairs():
    pfiles = collect_patient_variant_files(REPORTS_DIR)
    gtmap = {}
    gt_dir_path = Path(GT_DIR)
    # 递归查找所有符合命名规则的 JSON 文件
    for gt_file in gt_dir_path.glob("**/*_extracted_*.json"):
        match = re.match(r"^(?P<pid>.*)_extracted_(?P<variant>correct|wrong)\.json$", gt_file.name, flags=re.I)
        if match:
            gtmap[(match.group("pid"), match.group("variant").lower())] = str(gt_file)

    pairs = []
    for (pid, variant), files in pfiles.items():
        if (pid, variant) not in gtmap:
            continue
        endo_text = read_report_file(files.get("endoscopy"))
        path_text = read_report_file(files.get("pathology"))
        gt_flat = normalize_to_single(_read_json(gtmap[(pid, variant)]) or {})
        prompt = make_joint_prompt_single(pid=pid, variant=variant, endo_text=endo_text, path_text=path_text)
        target = format_target_json(pid, variant, gt_flat)
        pairs.append((pid, variant, prompt, target))
    return pairs

ALL_PAIRS = build_io_pairs()


In [ ]:

# -------------------------------- Tokenization & Data --------------------------------
def encode_example(prompt_text: str, target_text: str, tok) -> Dict[str, List[int]]:
    prompt_fmt = apply_chat(tok, prompt_text)
    prompt_ids = tok(prompt_fmt, add_special_tokens=False, truncation=True, max_length=MAX_LEN_INPUT)["input_ids"]
    target_ids = tok("\n" + target_text + (tok.eos_token or ""), add_special_tokens=False,
                     truncation=True, max_length=MAX_LEN_OUTPUT)["input_ids"]
    input_ids = (prompt_ids + target_ids)[:(MAX_LEN_INPUT+MAX_LEN_OUTPUT)]
    labels    = ([-100]*len(prompt_ids) + target_ids)[:(MAX_LEN_INPUT+MAX_LEN_OUTPUT)]
    attention_mask = [1]*len(input_ids)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

def to_hf_dataset(pairs, tok):
    # 将 pairs 转换为 Dataset，但不立即 tokenize
    dict_data = {
        "pid": [p[0] for p in pairs],
        "variant": [p[1] for p in pairs],
        "prompt": [p[2] for p in pairs],
        "target": [p[3] for p in pairs],
    }
    dataset = Dataset.from_dict(dict_data)
    # 使用 map 延迟 tokenize
    dataset = dataset.map(
        lambda x: encode_example(x["prompt"], x["target"], tok),
        remove_columns=["prompt", "target"],
        batched=False,
        load_from_cache_file=False,
    )
    return dataset

@dataclass
class LMDataCollator:
    tokenizer: AutoTokenizer
    pad_to_multiple_of: int = 8
    def __call__(self, features):
        batch = {}
        max_len = max(len(f["input_ids"]) for f in features)
        if self.pad_to_multiple_of:
            max_len = int(_math.ceil(max_len / self.pad_to_multiple_of) * self.pad_to_multiple_of)
        def pad(seq, pad_id, L): return seq + [pad_id]*(L - len(seq))
        pad_id = self.tokenizer.pad_token_id or self.tokenizer.eos_token_id
        batch["input_ids"] = [pad(f["input_ids"], pad_id, max_len) for f in features]
        batch["attention_mask"] = [pad(f["attention_mask"], 0, max_len) for f in features]
        batch["labels"] = [pad(f["labels"], -100, max_len) for f in features]
        return {k: torch.tensor(v) for k,v in batch.items()}

# -------------------------------- TrainingArguments --------------------------------
_TA_SIG = signature(TrainingArguments.__init__).parameters

def make_training_args(output_dir: str, bf16_ok: bool, fp16_ok: bool, has_val: bool) -> TrainingArguments:
    base = {
        "output_dir": os.path.join(output_dir, "checkpoints"),
        "per_device_eval_batch_size": 1,   # 显式设置
        "eval_accumulation_steps": 1,       # 减少累积步数，降低峰值
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "per_device_train_batch_size": BATCH_SIZE_TRAIN,
        "learning_rate": LR,
        "num_train_epochs": NUM_EPOCHS,
        "bf16": bf16_ok,
        "fp16": fp16_ok,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "logging_steps": LOG_STEPS,
        "save_steps": SAVE_STEPS,
        "eval_steps": EVAL_STEPS,
        "save_total_limit": 2,
        "report_to": [],
        "lr_scheduler_type": "cosine",
        "logging_strategy": "steps",
        "optim": "adamw_8bit",
        "evaluation_strategy": "epoch",   # 每个 epoch 后评估
        "save_strategy": "epoch",
        "load_best_model_at_end": True,   # 训练结束后加载最佳模型
        "metric_for_best_model": "eval_loss",
        "greater_is_better": False,
        "early_stopping_patience": 3,     # 早停 patience
        "early_stopping_threshold": 0.001,
        "dataloader_pin_memory": False,          # 减少CPU锁页内存，可能降低显存压力
    }

    # evaluation_strategy vs eval_strategy differences across versions
    if "evaluation_strategy" in _TA_SIG:
        base["evaluation_strategy"] = "epoch" if has_val else "no"
    elif "eval_strategy" in _TA_SIG:
        base["eval_strategy"] = "epoch" if has_val else "no"

    filtered = {k: v for k, v in base.items() if k in _TA_SIG}
    return TrainingArguments(**filtered)

# -------------------------------- Device/Precision/Quant --------------------------------
def bf16_supported() -> bool:
    return torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8

def _dtype():
    return torch.bfloat16 if bf16_supported() else torch.float16

bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=_dtype(),
)

# -------------------------------- Tokenizer/Model loaders --------------------------------
def load_tokenizer(repo_or_path: str):
    """Load tokenizer from cache if present; otherwise download if allowed."""
    kwargs = {
        "use_fast": True,
        "trust_remote_code": True,
        "cache_dir": HF_CACHE_DIR,
    }
    if not ALLOW_HF_DOWNLOAD:
        kwargs["local_files_only"] = True   # 禁止下载时，仅使用本地缓存

    tok = AutoTokenizer.from_pretrained(repo_or_path, **kwargs)

    if tok.pad_token_id is None and tok.eos_token_id is not None:
        tok.pad_token = tok.eos_token
    return tok



def _dtype_kwarg():
    sig = signature(AutoModelForCausalLM.from_pretrained).parameters    
    return ("dtype" if "dtype" in sig else "torch_dtype")

def load_model(repo_or_path: str):
    dtype_key = _dtype_kwarg()
    kwargs = {        
        "trust_remote_code": True,        
        "cache_dir": HF_CACHE_DIR,
        dtype_key: torch.float16,    
    }    
    if not ALLOW_HF_DOWNLOAD:
        kwargs["local_files_only"] = True
    model = AutoModelForCausalLM.from_pretrained(repo_or_path, **kwargs)    
    return model.to("cuda")

# -------------------------------- Inference + Eval --------------------------------

def _sha1_12(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()[:12]

def _write_jsonl(path: str, rec: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def _truncate_prompt_for_jsonformer(tok, prompt_text: str, max_tokens: int):
    """
    Jsonformer tokenizes internally; we must ensure the prompt won't exceed context.
    This truncates to max_tokens by keeping a 'head' and 'tail' slice.
    """
    ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
    n0 = len(ids)
    if n0 <= max_tokens:
        return prompt_text, n0, False

    head = min(512, max_tokens // 3)  # keep instructions/schema
    tail = max_tokens - head
    if tail <= 0:
        tail = max_tokens
        head = 0

    ids_trunc = ids[:head] + ids[-tail:]
    txt_trunc = tok.decode(ids_trunc, skip_special_tokens=False)
    return txt_trunc, n0, True

def _infer_and_eval(model, tok, tag_dir_root: str, test_pairs_list, label_tag: str):
    PRED_DIR  = os.path.join(tag_dir_root, "predictions_test")
    FIG_DIR   = os.path.join(tag_dir_root, "eval_figs")
    CONF_DIR  = os.path.join(tag_dir_root, "eval_confusions")
    TABLE_DIR = os.path.join(tag_dir_root, "eval_tables")
    for d in (PRED_DIR, FIG_DIR, CONF_DIR, TABLE_DIR):
        os.makedirs(d, exist_ok=True)

    # ---------------- Audit files ----------------
    AUDIT_JSONL = os.path.join(TABLE_DIR, "jsonformer_audit.jsonl")
    AUDIT_SUMMARY_JSON = os.path.join(TABLE_DIR, "jsonformer_audit_summary.json")

    # reset audit for this run
    if os.path.exists(AUDIT_JSONL):
        os.remove(AUDIT_JSONL)

    model.eval()
    if hasattr(model, "config"):
        model.config.use_cache = True

    if tok.pad_token_id is None and tok.eos_token_id is not None:
        tok.pad_token = tok.eos_token

    # ---------------- Build prompts ----------------
    prompts_fmt = []
    meta = []
    for (pid, variant, prompt, _) in test_pairs_list:
        meta.append((pid, variant))
        prompts_fmt.append(apply_chat(tok, prompt))

    # ---------------- Hard guarantee counters ----------------
    global JSONFORMER_CALLS
    start_calls = JSONFORMER_CALLS

    # Write audit header
    try:
        jf_ver = _md.version("jsonformer")
    except Exception:
        jf_ver = "unknown"
    try:
        tf_ver = _md.version("transformers")
    except Exception:
        tf_ver = "unknown"

    _write_jsonl(AUDIT_JSONL, {
        "type": "header",
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "label_tag": label_tag,
        "enforce_jsonformer": bool(ENFORCE_JSONFORMER),
        "schema_sig_combined": _schema_sig(COMBINED_SINGLE_SCHEMA),
        "schema_sig_inner": _schema_sig(ENDOSCOPY_SCHEMA),
        "jsonformer_max_string_token_length": JSONFORMER_MAX_STRING_TOKEN_LENGTH,
        "MAX_LEN_INPUT": MAX_LEN_INPUT,
        "jsonformer_version": jf_ver,
        "transformers_version": tf_ver,
        "model_name_or_path": getattr(getattr(model, "config", None), "_name_or_path", "unknown"),
    })

    # ---------------- Extraction loop (Jsonformer-only) ----------------
    n_total = len(prompts_fmt)
    n_used_jsonformer = 0
    n_truncated = 0
    total_s = 0.0

    total_raw_validator_errs = 0
    total_post_validator_errs = 0
    total_missing_keys = 0
    total_sanitize_changes = 0

    for (pid, variant), prompt_fmt in zip(meta, prompts_fmt):
        if not ENFORCE_JSONFORMER:
            raise RuntimeError("ENFORCE_JSONFORMER is False but Jsonformer-only path is required for strict enforcement.")

        prompt_trunc, n_tokens_orig, was_trunc = _truncate_prompt_for_jsonformer(tok, prompt_fmt, MAX_LEN_INPUT)
        if was_trunc:
            n_truncated += 1

        t0 = time.time()
        raw = run_jsonformer(
            tok, model, COMBINED_SINGLE_SCHEMA, prompt_trunc,
            max_string_token_length=JSONFORMER_MAX_STRING_TOKEN_LENGTH
        )
        dt = time.time() - t0
        total_s += dt
        n_used_jsonformer += 1

        # ----- Structural checks on Jsonformer output -----
        if not isinstance(raw, dict):
            raise RuntimeError(f"[STRICT] Jsonformer returned non-dict for {pid}:{variant}: {type(raw)}")

        if "endoscopy and pathology" not in raw or not isinstance(raw["endoscopy and pathology"], dict):
            raise RuntimeError(
                f"[STRICT] Jsonformer output missing 'endoscopy and pathology' for {pid}:{variant}. "
                f"Keys={list(raw.keys())}"
            )

        ep_raw = raw["endoscopy and pathology"]

        # Missing key count (should be 0 if Jsonformer fully follows ENDOSCOPY_SCHEMA.required)
        missing_keys = sum(1 for k in REQUIRED_KEYS if k not in ep_raw)
        total_missing_keys += missing_keys

        # Validate RAW (before sanitize) on a filled dict so missing keys don't spam errors
        ep_raw_filled = _ensure_all_keys(ep_raw)
        raw_errs = list(ENDO_VALIDATOR.iter_errors(ep_raw_filled))
        total_raw_validator_errs += len(raw_errs)

        # Sanitize (keeps your current normalization logic)
        ep_clean = validate_and_sanitize(ep_raw, ENDO_VALIDATOR)

        # Validate AFTER sanitize (should be 0)
        post_errs = list(ENDO_VALIDATOR.iter_errors(ep_clean))
        total_post_validator_errs += len(post_errs)

        # Count how many fields changed due to coercion/sanitize
        sanitize_changes = sum(
            1 for k in REQUIRED_KEYS
            if str(ep_raw.get(k, "not_mentioned")).strip() != str(ep_clean.get(k, "not_mentioned")).strip()
        )
        total_sanitize_changes += sanitize_changes

        # 记录原始输出（清洗前）
        raw_inner = raw.get("endoscopy and pathology", {})
        raw_record = {
            "pid": pid,
            "variant": variant,
            "raw_json": json.dumps(raw_inner, ensure_ascii=False)   # 整体存为 JSON 字符串
        }
        # 或者更细粒度：每个字段单独保存
        # 这里我们写入一个专门的 CSV 文件，便于后续分析
        import csv
        raw_csv_path = os.path.join(TABLE_DIR, "raw_outputs.csv")
        # 首次写入表头
        if not os.path.exists(raw_csv_path):
            with open(raw_csv_path, "w", newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(["pid", "variant"] + REQUIRED_KEYS)
        # 追加一行
        with open(raw_csv_path, "a", newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            row = [pid, variant] + [raw_inner.get(k, "MISSING") for k in REQUIRED_KEYS]
            writer.writerow(row)

        
        out_obj = {"patient_id": pid, "variant": variant, "endoscopy and pathology": ep_clean}
        out_path = os.path.join(PRED_DIR, f"{pid}_{variant}_extracted_pred.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(out_obj, f, ensure_ascii=False, indent=2)

        # Per-sample audit record
        _write_jsonl(AUDIT_JSONL, {
            "type": "sample",
            "pid": pid,
            "variant": variant,
            "prompt_hash": _sha1_12(prompt_trunc),
            "prompt_tokens_orig": int(n_tokens_orig),
            "prompt_truncated": bool(was_trunc),
            "used_jsonformer": True,
            "jsonformer_elapsed_s": round(dt, 4),
            "raw_patient_id": raw.get("patient_id", None),
            "raw_variant": raw.get("variant", None),
            "missing_required_keys_in_raw_inner": int(missing_keys),
            "raw_validator_errors": int(len(raw_errs)),
            "post_sanitize_validator_errors": int(len(post_errs)),
            "sanitize_changes_count": int(sanitize_changes),
        })

        print(f"[PRED-{label_tag}][JSONFORMER] {pid} [{variant}] -> {out_path} | dt={dt:.2f}s | trunc={was_trunc} | miss={missing_keys} | raw_errs={len(raw_errs)}")

    # ---------------- Enforcement assertions ----------------
    calls_used = JSONFORMER_CALLS - start_calls

    # Jsonformer was called exactly once per sample
    if calls_used != n_total:
        raise RuntimeError(f"[STRICT] Jsonformer calls mismatch: calls_used={calls_used} vs n_total={n_total}")

    # Loop path used Jsonformer for all samples
    if n_used_jsonformer != n_total:
        raise RuntimeError(f"[STRICT] used_jsonformer mismatch: {n_used_jsonformer}/{n_total}")

    # ---------------- Write audit summary ----------------
    summary = {
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "label_tag": label_tag,
        "n_total": int(n_total),
        "n_used_jsonformer": int(n_used_jsonformer),
        "pct_jsonformer": float(n_used_jsonformer / max(1, n_total)),
        "jsonformer_calls_used": int(calls_used),
        "n_prompt_truncated": int(n_truncated),
        "mean_jsonformer_time_s": float(total_s / max(1, n_total)),
        "total_raw_validator_errors": int(total_raw_validator_errs),
        "total_post_sanitize_validator_errors": int(total_post_validator_errs),
        "total_missing_required_keys_in_raw_inner": int(total_missing_keys),
        "total_sanitize_changes": int(total_sanitize_changes),
        "schema_sig_combined": _schema_sig(COMBINED_SINGLE_SCHEMA),
        "schema_sig_inner": _schema_sig(ENDOSCOPY_SCHEMA),
        "audit_jsonl": AUDIT_JSONL,
    }
    with open(AUDIT_SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\n[AUDIT] Jsonformer STRICT enforcement PASSED.")
    print(f"[AUDIT] Jsonformer calls used: {calls_used}/{n_total} (must be equal)")
    print(f"[AUDIT] Audit log      : {AUDIT_JSONL}")
    print(f"[AUDIT] Audit summary  : {AUDIT_SUMMARY_JSON}")

    # ---------------- Evaluate ----------------
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    eval_result = evaluate_and_plot(gt_dir=GT_DIR, pred_dir=PRED_DIR, fig_dir=FIG_DIR, conf_dir=CONF_DIR)

    feature_scores = eval_result["feature_scores"].sort_values("accuracy", ascending=False).reset_index(drop=True)
    patient_scores = eval_result["patient_scores"]
    raw_long       = eval_result["raw"]

    # 新增：保存总体指标（含非空准确率）
    overall_metrics_df = pd.DataFrame([{
        "overall_accuracy": eval_result["overall_accuracy"],
        "non_null_overall_accuracy": eval_result["non_null_overall_accuracy"]
    }])
    overall_metrics_df.to_csv(os.path.join(TABLE_DIR, "overall_metrics.csv"), index=False)

    feature_scores.to_csv(os.path.join(TABLE_DIR, "feature_scores.csv"), index=False)
    patient_scores.to_csv(os.path.join(TABLE_DIR, "patient_scores.csv"), index=False)
    raw_long.to_csv(os.path.join(TABLE_DIR, "all_comparisons_long.csv"), index=False)

    return eval_result, feature_scores, patient_scores, raw_long, PRED_DIR, FIG_DIR, CONF_DIR, TABLE_DIR

In [ ]:

# -------------------------------- Train/Eval for a single base --------------------------------
def train_eval_one(BASE_MODEL_KEY: str):
    base_model_id = MODEL_CANDIDATES[BASE_MODEL_KEY]
    print(f"\n========== [{BASE_MODEL_KEY}] {base_model_id} ==========")

    # Use the global run-mode flags defined above
    global DO_ZERO_SHOT, DO_FINE_TUNE

    OUTPUT_DIR  = os.path.join(BASE_DIR, f"{BASE_MODEL_KEY}_finetune_{RUN_TAG}")
    ZS_DIR_ROOT = os.path.join(OUTPUT_DIR, "zero_shot")
    FT_DIR_ROOT = os.path.join(OUTPUT_DIR, "finetuned")
    ADAPTER_DIR = os.path.join(FT_DIR_ROOT, "lora_adapter")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if DO_ZERO_SHOT:
        os.makedirs(ZS_DIR_ROOT, exist_ok=True)
    if DO_FINE_TUNE:
        os.makedirs(FT_DIR_ROOT, exist_ok=True)

    # Ensure local availability (optional fetch if allowed)
    load_path = base_model_id

    # Tokenizer (needed for prompts in both ZS and FT)
    tok = load_tokenizer(load_path)
    if tok.pad_token_id is None and tok.eos_token_id is not None:
        tok.pad_token = tok.eos_token

    results = []

    # ---------------- Zero-shot ----------------
    if DO_ZERO_SHOT:
        print(f"\n[ZERO-SHOT] Loading base model for zero-shot inference: {base_model_id}")
    # 使用 4-bit 量化加载
        from transformers import BitsAndBytesConfig
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if bf16_supported() else torch.float16
        )
        base_zs = AutoModelForCausalLM.from_pretrained(
            load_path,
            quantization_config=bnb_config,
            device_map="auto",          # 自动分配到 GPU
            trust_remote_code=True
        )
        base_zs.resize_token_embeddings(len(tok))
        base_zs.eval()

        zs_eval_result, zs_feature_scores, zs_patient_scores, zs_raw_long, PRED_DIR_ZS, FIG_DIR_ZS, CONF_DIR_ZS, TABLE_DIR_ZS = \
            _infer_and_eval(base_zs, tok, ZS_DIR_ROOT, test_pairs, label_tag="ZS")

        zs_overall_acc = float(zs_eval_result["overall_accuracy"])
        zs_non_null_acc = float(zs_eval_result.get("non_null_overall_accuracy", float('nan')))  # <--- 新增
        zs_null_acc = float(zs_eval_result.get("null_accuracy", float('nan')))          # <--- 新增
        zs_balanced_acc = float(zs_eval_result.get("balanced_accuracy", float('nan')))  # <--- 新增
        print(f"[ZERO-SHOT] Overall accuracy (test, {int(TEST_FRAC*100)}%): {zs_overall_acc:.3f}")
        print(f"[ZERO-SHOT] Non-null accuracy: {zs_non_null_acc:.3f}")  # <--- 新增（可选打印）

        results.append({
            "model_key": f"{BASE_MODEL_KEY}_zs",
            "model_id": base_model_id,
            "overall_accuracy": zs_overall_acc,
            "non_null_overall_accuracy": zs_non_null_acc,  # <--- 新增
            "null_accuracy": zs_null_acc,                # <--- 新增
            "balanced_accuracy": zs_balanced_acc,        # <--- 新增
            "feature_scores": zs_feature_scores.copy(),
            "output_dir": OUTPUT_DIR,
            "tables_dir": TABLE_DIR_ZS,
            "figs_dir": FIG_DIR_ZS,
        })

        # Free VRAM
        del base_zs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print("[ZERO-SHOT] Skipped (EXPERIMENT_MODE)")

    # ---------------- Fine-tune (QLoRA) + inference ----------------
    if DO_FINE_TUNE:
        # Build datasets ONLY if training is enabled
        ds_train = to_hf_dataset(train_pairs, tok)
        ds_val   = to_hf_dataset(val_pairs, tok) if len(val_pairs) > 0 else None
        collator = LMDataCollator(tokenizer=tok)

        print(f"\n[FINE-TUNE] Preparing base model + LoRA: {base_model_id}")
        
        from transformers import BitsAndBytesConfig
        bnb_config = BitsAndBytesConfig(
          load_in_4bit=True,
          bnb_4bit_use_double_quant=True,
          bnb_4bit_quant_type="nf4",
          bnb_4bit_compute_dtype=torch.bfloat16 if bf16_supported() else torch.float16
        )
        base_model = AutoModelForCausalLM.from_pretrained(
          load_path,
          quantization_config=bnb_config,
          device_map="auto",
          trust_remote_code=True
        )
        base_model.resize_token_embeddings(len(tok))
        base_model = prepare_model_for_kbit_training(base_model)

        base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        base_model.config.use_cache = False  # important for training

        _lora_sig = signature(LoraConfig.__init__).parameters

        lora_kwargs = dict(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=TARGET_MODULES,
            task_type="CAUSAL_LM",
        )
        if "use_rslora" in _lora_sig:
            lora_kwargs["use_rslora"] = True

        lora_cfg = LoraConfig(**lora_kwargs)
        model = get_peft_model(base_model, lora_cfg)
        model.print_trainable_parameters()

        set_seed_all(SEED)
        steps_per_epoch = max(1, len(ds_train) // (BATCH_SIZE_TRAIN * GRAD_ACCUM_STEPS))
        print(f"[INFO] Approx steps/epoch: {steps_per_epoch}")

        train_args = make_training_args(
            output_dir=FT_DIR_ROOT,
            bf16_ok=bf16_supported(),
            fp16_ok=(not bf16_supported()),
            has_val=(ds_val is not None and len(ds_val) > 0)
        )

        trainer = Trainer(
            model=model,
            args=train_args,
            train_dataset=ds_train,
            eval_dataset=ds_val,
            data_collator=collator,
            callbacks=[
                EarlyStoppingCallback(
                    early_stopping_patience=3,
                    early_stopping_threshold=0.001,
                )
            ],
        )
        trainer.train()
        torch.cuda.synchronize()   # 等待所有CUDA核完成
        torch.cuda.empty_cache()   # 清空缓存
        trainer.save_model(ADAPTER_DIR)
        tok.save_pretrained(ADAPTER_DIR)

        # 绘制损失曲线并保存
        log_history = trainer.state.log_history

        eval_epochs = []
        eval_losses = []
        train_epochs = []
        train_losses = []
        epoch_train_loss = {}
        epoch_train_count = {}

        for log in log_history:
            if "eval_loss" in log:
                epoch = log.get("epoch")
                if epoch is not None:
                    eval_epochs.append(epoch)
                    eval_losses.append(log["eval_loss"])
            if "loss" in log and "epoch" in log:
                epoch = log["epoch"]
                loss = log["loss"]
                epoch_train_loss[epoch] = epoch_train_loss.get(epoch, 0) + loss
                epoch_train_count[epoch] = epoch_train_count.get(epoch, 0) + 1
        
        for epoch in sorted(epoch_train_loss.keys()):
            avg_loss = epoch_train_loss[epoch] / epoch_train_count[epoch]
            train_epochs.append(epoch)
            train_losses.append(avg_loss)

        if eval_epochs:
            plt.figure(figsize=(10, 6))
            if train_epochs:
                plt.plot(train_epochs, train_losses, 'o-', label='Training Loss (avg per epoch)', color='blue')
            plt.plot(eval_epochs, eval_losses, 's-', label='Validation Loss', color='red')
            best_idx = np.argmin(eval_losses)
            best_epoch = eval_epochs[best_idx]
            best_loss = eval_losses[best_idx]
            plt.scatter(best_epoch, best_loss, color='green', s=100, zorder=5, label=f'Best Epoch {best_epoch:.1f} (loss={best_loss:.4f})')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.title('Training and Validation Loss per Epoch')
            plt.legend()
            plt.grid(True)
            loss_curve_path = os.path.join(FT_DIR_ROOT, 'loss_curve.png')
            plt.savefig(loss_curve_path, dpi=150, bbox_inches='tight')
            plt.close()
            print(f"[INFO] Loss curve saved to {loss_curve_path}")

            
        # Free training objects (important: base_model can still hold VRAM if not deleted)
        del trainer, model, base_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # ---------------- Inference with adapter ----------------
        print(f"\n[INFER-FT] Loading base + adapter for inference.")
        tok_inf = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True, trust_remote_code=True)
        if tok_inf.pad_token_id is None and tok_inf.eos_token_id is not None:
          tok_inf.pad_token = tok_inf.eos_token
            
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if bf16_supported() else torch.float16
        )

        base_inf = AutoModelForCausalLM.from_pretrained(
            load_path,
            quantization_config=bnb_config,
            device_map="auto",          # 自动分配到 GPU，无需手动 .to("cuda")
            trust_remote_code=True
        )
        base_inf.resize_token_embeddings(len(tok_inf))
        model_inf = PeftModel.from_pretrained(base_inf, ADAPTER_DIR)
        model_inf.eval()

        ft_eval_result, ft_feature_scores, ft_patient_scores, ft_raw_long, PRED_DIR_FT, FIG_DIR_FT, CONF_DIR_FT, TABLE_DIR_FT = \
            _infer_and_eval(model_inf, tok_inf, FT_DIR_ROOT, test_pairs, label_tag="FT")

        ft_overall_acc = float(ft_eval_result["overall_accuracy"])
        ft_non_null_acc = float(ft_eval_result.get("non_null_overall_accuracy", float('nan')))  # <--- 新增
        ft_null_acc = float(ft_eval_result.get("null_accuracy", float('nan')))          # <--- 新增
        ft_balanced_acc = float(ft_eval_result.get("balanced_accuracy", float('nan')))  # <--- 新增
        print(f"[FINE-TUNE] Overall accuracy (test, {int(TEST_FRAC*100)}%): {ft_overall_acc:.3f}")
        print(f"[FINE-TUNE] Non-null accuracy: {ft_non_null_acc:.3f}")  # <--- 新增（可选打印）

        results.append({
            "model_key": f"{BASE_MODEL_KEY}_ft",
            "model_id": base_model_id,
            "overall_accuracy": ft_overall_acc,
            "non_null_overall_accuracy": ft_non_null_acc,  # <--- 新增
            "null_accuracy": ft_null_acc,                
            "balanced_accuracy": ft_balanced_acc,
            "feature_scores": ft_feature_scores.copy(),
            "output_dir": OUTPUT_DIR,
            "tables_dir": TABLE_DIR_FT,
            "figs_dir": FIG_DIR_FT,
        })
        # Free inference objects
        del base_inf, model_inf
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    else:
        print("[FINE-TUNE] Skipped (EXPERIMENT_MODE)")

    if not results:
        raise RuntimeError("Nothing executed. Set EXPERIMENT_MODE to 'zero_shot', 'fine_tune', or 'both'.")

    print("\n================== Completed ==================")
    print(f"Base model: {BASE_MODEL_KEY} → {base_model_id}")
    if DO_ZERO_SHOT:
        print(f"[ZS] Output: {ZS_DIR_ROOT}")
    if DO_FINE_TUNE:
        print(f"[FT] Adapter: {ADAPTER_DIR}")
        print(f"[FT] Output: {FT_DIR_ROOT}")

    return results

In [ ]:

# -------------------------------- Comparison plots (optional) -------------------------------
def make_comparison_plots(results_list: List[dict], run_tag: str):
    if len(results_list) < 2:
        print("[INFO] Need at least two model variants to compare; skipping combined plots.")
        return None
    cmp_dir = os.path.join(BASE_DIR, f"comparison_{run_tag}")
    os.makedirs(cmp_dir, exist_ok=True)

    # Overall accuracy bar
    model_labels = [r["model_key"] for r in results_list]
    overall_vals = [r["overall_accuracy"] for r in results_list]
    plt.figure(figsize=(6 + 2*len(results_list), 5))
    x = np.arange(len(model_labels)); plt.bar(x, overall_vals)
    plt.xticks(x, model_labels, rotation=30, ha="right")
    plt.ylabel(f"Overall Accuracy (Test {int(TEST_FRAC*100)}%)"); plt.title("Overall Accuracy: Zero-Shot vs Fine-Tuned")
    plt.ylim(0, 1.0)
    for i, v in enumerate(overall_vals): plt.text(i, min(0.99, v + 0.02), f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    overall_path = os.path.join(cmp_dir, f"overall_accuracy_{run_tag}.png")
    plt.tight_layout(); plt.savefig(overall_path, dpi=160); plt.close()
    print(f"[CMP] Saved overall accuracy comparison → {overall_path}")


    # Per-feature grouped bars
    dfs = []
    for r in results_list:
        fcol = _feature_col(r["feature_scores"])
        df = (r["feature_scores"].rename(columns={fcol: "feature"})[["feature","accuracy"]].set_index("feature"))
        df.columns = [r["model_key"]]; dfs.append(df)
    merged = pd.concat(dfs, axis=1, sort=False).fillna(0.0)
    merged["mean_acc"] = merged.mean(axis=1)
    merged_sorted = merged.sort_values("mean_acc", ascending=False).drop(columns=["mean_acc"])
    TOP_N = min(39, len(merged_sorted)); merged_top = merged_sorted.head(TOP_N)
    csv_path = os.path.join(cmp_dir, f"feature_accuracy_comparison_top{TOP_N}_{run_tag}.csv")
    merged_top.to_csv(csv_path); print(f"[CMP] Saved per-feature accuracy table → {csv_path}")

    plt.figure(figsize=(max(12, TOP_N * 0.6), 6))
    idx = np.arange(len(merged_top.index)); width = 0.8 / len(results_list)
    for i, r in enumerate(results_list):
        plt.bar(idx + i*width, merged_top[r["model_key"]].values, width=width, label=r["model_key"])
    plt.xticks(idx + (len(results_list)-1)*width/2, merged_top.index, rotation=60, ha="right")
    plt.ylabel("Accuracy"); plt.title(f"Per-Feature Accuracy (Top {TOP_N}) — Zero-Shot vs Fine-Tuned")
    plt.ylim(0, 1.0); plt.legend(ncol=min(3, len(results_list)))
    feat_path = os.path.join(cmp_dir, f"feature_accuracy_side_by_side_top{TOP_N}_{run_tag}.png")
    plt.tight_layout(); plt.savefig(feat_path, dpi=160); plt.close()
    print(f"[CMP] Saved per-feature side-by-side plot → {feat_path}")
    # ---- 新增：非空准确率对比 ----
    non_null_vals = [r.get("non_null_overall_accuracy", float('nan')) for r in results_list]
    plt.figure(figsize=(6 + 2*len(results_list), 5))
    x = np.arange(len(model_labels))
    plt.bar(x, non_null_vals, color='orange')
    plt.xticks(x, model_labels, rotation=30, ha="right")
    plt.ylabel("Non-null Overall Accuracy")
    plt.title("Non-Null Accuracy: Zero-Shot vs Fine-Tuned")
    plt.ylim(0, 1.0)
    for i, v in enumerate(non_null_vals):
        if not np.isnan(v):
            plt.text(i, min(0.99, v + 0.02), f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    nonnull_path = os.path.join(cmp_dir, f"non_null_accuracy_{run_tag}.png")
    plt.tight_layout()
    plt.savefig(nonnull_path, dpi=160)
    plt.close()
    print(f"[CMP] Saved non-null accuracy comparison → {nonnull_path}")
    return cmp_dir

def set_data_split(seed: int):
    global train_pairs, val_pairs, test_pairs, RUN_TAG
    set_seed_all(seed)
    pairs = ALL_PAIRS[:]
    if MAX_SAMPLES is not None:
        pairs = pairs[:MAX_SAMPLES]
    if not pairs:
        raise SystemExit("[ERROR] No (prompt, target) pairs found for this run.")

    # 1. 提取所有唯一的患者 ID
    patient_ids = list(set(pid for pid, _, _, _ in pairs))
    random.shuffle(patient_ids)

    # 2. 按比例划分患者 ID
    n_total_patients = len(patient_ids)
    n_train_patients = int(n_total_patients * TRAIN_FRAC)
    n_val_patients = int(n_total_patients * VAL_FRAC)

    train_patient_set = set(patient_ids[:n_train_patients])
    val_patient_set = set(patient_ids[n_train_patients:n_train_patients + n_val_patients])
    test_patient_set = set(patient_ids[n_train_patients + n_val_patients:])

    # 3. 根据患者 ID 分配样本对
    train_pairs = [p for p in pairs if p[0] in train_patient_set]
    val_pairs   = [p for p in pairs if p[0] in val_patient_set]
    test_pairs  = [p for p in pairs if p[0] in test_patient_set]

    # 检查集合非空
    if not train_pairs or not test_pairs:
        raise SystemExit("[ERROR] One of the splits (train/test) is empty. Adjust TRAIN_FRAC/TEST_FRAC.")

    RUN_TAG = datetime.datetime.now().strftime(f"sm_ft_s{seed}_%Y%m%d_%H%M%S")
    print(f"\n[RUN-SETUP] seed={seed} | patients: train={len(train_patient_set)} val={len(val_patient_set)} test={len(test_patient_set)} | samples: train={len(train_pairs)} val={len(val_pairs)} test={len(test_pairs)} / total={len(pairs)}")
    print(f"[RUN-SETUP] RUN_TAG={RUN_TAG}")
    print(f"[INFO] Schema sigs → child={_schema_sig(ENDOSCOPY_SCHEMA)} | combined={_schema_sig(COMBINED_SINGLE_SCHEMA)}")

all_runs_results = []

for run_idx, seed_i in enumerate(RUN_SEEDS, start=1):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
    print(f"\n======================== RUN {run_idx}/{N_RUNS} (seed={seed_i}) ========================")
    set_data_split(seed_i)

    results_this_run = []
    for _mk in MODELS_TO_RUN:
        res_list = train_eval_one(_mk)
        for d in res_list:
            dd = d.copy(); dd["run"] = run_idx; dd["seed"] = seed_i
            all_runs_results.append(dd)
            results_this_run.append(d)

    make_comparison_plots(results_this_run, RUN_TAG)

# ==================== Aggregation: mean ± std across runs ====================
overall_rows = [{"run": d["run"], "seed": d["seed"], "model_key": d["model_key"],
                 "model_id": d["model_id"], 
                 "overall_accuracy": float(d["overall_accuracy"]),
                 "non_null_overall_accuracy": float(d.get("non_null_overall_accuracy", float('nan'))),
                 "null_accuracy": float(d.get("null_accuracy", float('nan'))),                # <--- 新增
                 "balanced_accuracy": float(d.get("balanced_accuracy", float('nan')))}        # <--- 新增
                for d in all_runs_results]
overall_df = pd.DataFrame(overall_rows)
overall_summary = (overall_df.groupby(["model_key","model_id"], as_index=False)
                   .agg(mean_accuracy=("overall_accuracy","mean"),
                        std_accuracy=("overall_accuracy","std"),
                        mean_non_null_accuracy=("non_null_overall_accuracy","mean"),
                        std_non_null_accuracy=("non_null_overall_accuracy","std"),
                        mean_null_accuracy=("null_accuracy","mean"),                # <--- 新增
                        std_null_accuracy=("null_accuracy","std"),                  # <--- 新增
                        mean_balanced_accuracy=("balanced_accuracy","mean"),        # <--- 新增
                        std_balanced_accuracy=("balanced_accuracy","std"),          # <--- 新增
                        n_runs=("overall_accuracy","count"))
                   .sort_values(["model_key"]))  
AGG_DIR = os.path.join(BASE_DIR, f"aggregated_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}")
os.makedirs(AGG_DIR, exist_ok=True)
overall_df.to_csv(os.path.join(AGG_DIR, "overall_accuracy_runs.csv"), index=False)
overall_summary.to_csv(os.path.join(AGG_DIR, "overall_accuracy_mean_std.csv"), index=False)
print(f"[AGG] Saved overall accuracy runs → {os.path.join(AGG_DIR, 'overall_accuracy_runs.csv')}")
print(f"[AGG] Saved overall mean±std → {os.path.join(AGG_DIR, 'overall_accuracy_mean_std.csv')}")

for model_key in sorted(set(r["model_key"] for r in all_runs_results)):
    feat_records = []
    for d in all_runs_results:
        if d["model_key"] != model_key: continue
        df_fs = d["feature_scores"].copy()
        fcol = _feature_col(df_fs)
        if "accuracy" not in df_fs.columns:
            raise ValueError(f"[AGG] feature_scores for {model_key} lacks 'accuracy' column.")
        df_fs = df_fs.rename(columns={fcol: "feature"})[["feature","accuracy"]]
        df_fs["run"] = d["run"]; df_fs["seed"] = d["seed"]
        feat_records.append(df_fs)
    if not feat_records: continue
    feats_all = pd.concat(feat_records, axis=0, ignore_index=True)
    feats_summary = (feats_all.groupby("feature", as_index=False)
                     .agg(mean_accuracy=("accuracy","mean"),
                          std_accuracy=("accuracy","std"),
                          n_runs=("accuracy","count"))
                     .sort_values("mean_accuracy", ascending=False))
    out_runs = os.path.join(AGG_DIR, f"{model_key}_feature_accuracy_runs.csv")
    out_mean = os.path.join(AGG_DIR, f"{model_key}_feature_accuracy_mean_std.csv")
    feats_all.to_csv(out_runs, index=False); feats_summary.to_csv(out_mean, index=False)
    print(f"[AGG] {model_key}: saved per-feature runs → {out_runs}")
    print(f"[AGG] {model_key}: saved per-feature mean±std → {out_mean}")

try:
    plt.figure(figsize=(max(8, 2.2*len(overall_summary)), 5))
    x = np.arange(len(overall_summary))
    means = overall_summary["mean_accuracy"].values
    stds  = overall_summary["std_accuracy"].fillna(0.0).values
    plt.bar(x, means, yerr=stds, capsize=5)
    plt.xticks(x, overall_summary["model_key"], rotation=30, ha="right")
    plt.ylabel("Overall Accuracy (mean ± std)"); plt.ylim(0, 1.0)
    for i, (m, s) in enumerate(zip(means, stds)):
        plt.text(i, min(0.99, m + (0 if np.isnan(s) else s) + 0.03),
                 f"{m:.3f}±{(0 if np.isnan(s) else s):.3f}", ha="center", va="bottom", fontsize=9)
    plt.title(f"Overall Accuracy across {N_RUNS} runs")
    out_plot = os.path.join(AGG_DIR, f"overall_accuracy_mean_std_{N_RUNS}runs.png")
    plt.tight_layout(); plt.savefig(out_plot, dpi=160); plt.close()
    print(f"[AGG] Saved overall mean±std plot → {out_plot}")    
except Exception as e:
    print(f"[AGG] Plotting failed (non-critical): {e}")

# ---- 新增：总体、非空、平衡准确率三组对比柱状图 ----
try:
    model_keys = overall_summary["model_key"].values
    n_models = len(model_keys)
    x = np.arange(n_models)
    width = 0.25  # 三组柱子，宽度调窄一点

    means_overall = overall_summary["mean_accuracy"].values
    stds_overall = overall_summary["std_accuracy"].fillna(0.0).values
    means_nonnull = overall_summary["mean_non_null_accuracy"].values
    stds_nonnull = overall_summary["std_non_null_accuracy"].fillna(0.0).values
    means_balanced = overall_summary["mean_balanced_accuracy"].values
    stds_balanced = overall_summary["std_balanced_accuracy"].fillna(0.0).values

    plt.figure(figsize=(max(8, 2.2 * n_models), 5))
    
    # 第一组：总体准确率（蓝色）
    rects1 = plt.bar(x - width, means_overall, width, yerr=stds_overall, 
                     capsize=5, label='Overall Accuracy', color='blue', alpha=0.7)
    # 第二组：非空准确率（橙色）
    rects2 = plt.bar(x, means_nonnull, width, yerr=stds_nonnull, 
                     capsize=5, label='Non-null Accuracy', color='orange', alpha=0.7)
    # 第三组：平衡准确率（绿色）
    rects3 = plt.bar(x + width, means_balanced, width, yerr=stds_balanced, 
                     capsize=5, label='Balanced Accuracy', color='green', alpha=0.7)

    plt.xlabel('Model')
    plt.ylabel('Accuracy (mean ± std)')
    plt.xticks(x, model_keys, rotation=30, ha='right')
    plt.ylim(0, 1.0)
    plt.title(f'Overall vs Non-null vs Balanced Accuracy across {N_RUNS} runs')
    plt.legend()

    # 在柱子上添加数值标签
    for i, (m1, s1, m2, s2, m3, s3) in enumerate(zip(
            means_overall, stds_overall, 
            means_nonnull, stds_nonnull, 
            means_balanced, stds_balanced)):
        # 总体准确率标签
        plt.text(i - width, min(0.99, m1 + s1 + 0.02), f'{m1:.3f}', 
                 ha='center', va='bottom', fontsize=8)
        # 非空准确率标签（跳过 NaN）
        if not np.isnan(m2):
            plt.text(i, min(0.99, m2 + s2 + 0.02), f'{m2:.3f}', 
                     ha='center', va='bottom', fontsize=8)
        # 平衡准确率标签（跳过 NaN）
        if not np.isnan(m3):
            plt.text(i + width, min(0.99, m3 + s3 + 0.02), f'{m3:.3f}', 
                     ha='center', va='bottom', fontsize=8)

    out_compare_plot = os.path.join(AGG_DIR, 
                                    f"overall_vs_nonnull_vs_balanced_accuracy_{N_RUNS}runs.png")
    plt.tight_layout()
    plt.savefig(out_compare_plot, dpi=160)
    plt.close()
    print(f"[AGG] Saved overall vs non-null vs balanced accuracy comparison plot → {out_compare_plot}")
except Exception as e:
    print(f"[AGG] Comparison plot failed (non-critical): {e}")

print(f"\n[AGG] All aggregation artifacts stored in: {AGG_DIR}")

# 清理模型缓存
if os.path.exists(HF_CACHE_DIR):
    import shutil
    shutil.rmtree(HF_CACHE_DIR)
    print("[CLEANUP] Deleted HF cache")